# Pruebas Nahu: exploración (ejercicio 1) y limpieza (ejercicio 2)

Cada celda de código tiene arriba un texto corto que explica qué hace y para qué sirve.

In [ ]:
# importar librerías
from wget import download
from os import path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_colwidth", None)
sns.set_theme(style="whitegrid")

## 1. Descarga y apertura de los datos

Se descargan los dos archivos solo si no están ya en la carpeta (así no se vuelven a bajar los 143 MB cada vez).

In [9]:

if not path.exists("schema.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/refs/heads/main/packages/archive/2025/schema.csv")
if not path.exists("results.csv"):
  download("https://media.githubusercontent.com/media/StackExchange/Survey/6ae2209cd35a55e92739ba82f7887fb86807a069/packages/archive/2025/results.csv")

Se abren los archivos con `pd.read_csv`. `results.csv` queda en `df` (las respuestas de la encuesta) y `schema.csv` en `schema` (la descripción de cada pregunta). `low_memory=False` hace que pandas lea cada columna completa antes de decidir su tipo y evita el `DtypeWarning` de las columnas con tipos mezclados.

In [ ]:
df = pd.read_csv("results.csv", low_memory=False)
schema = pd.read_csv("schema.csv")

In [ ]:
print(df.shape)
df.info()

In [7]:
df.head()

,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",None of the above,10.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code;GitHub Copilot;Google Gemini,NaN,When I don’t trust AI’s answers;When I want to...,"Understand how things actually work, problem s...",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,None of the above,4.0,"Yes, I am not new to coding but am learning ne...","Other online resources (e.g. standard search, ...","Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,ChatGPT;Claude Code,NaN,When I don’t trust AI’s answers;When I want to...,NaN,36197.0,6.0
4,5,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...","Caring for dependents (children, elderly, etc.)",21.0,"No, I am not new to coding and did not learn n...",NaN,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers,"critical thinking, the skill to define the tas...",60000.0,7.0


In [11]:
schema.shape

(139, 6)

In [12]:
schema.head()

,qid,qname,question,type,sub,sq_id
0,QID18,TechEndorse_1,What attracts you to a technology or causes you to endorse it (most to least important)?,RO,AI integration or AI Agent capabilities,1.0
1,QID18,TechEndorse_2,What attracts you to a technology or causes you to endorse it (most to least important)?,RO,Easy-to-use API,2.0
2,QID18,TechEndorse_3,What attracts you to a technology or causes you to endorse it (most to least important)?,RO,Robust and complete API,3.0
3,QID18,TechEndorse_4,What attracts you to a technology or causes you to endorse it (most to least important)?,RO,Customizable and manageable codebase,4.0
4,QID18,TechEndorse_5,What attracts you to a technology or causes you to endorse it (most to least important)?,RO,Reputation for quality,5.0


`value_counts()` cuenta cuántas veces aparece cada valor de una columna. Acá cuenta cuántas filas del esquema hay de cada tipo de pregunta (`RO` ranking, `MC` opción múltiple, `TE` texto o número, `Matrix` matriz).

In [10]:
schema["type"].value_counts()

type
RO        49
MC        48
TE        29
Matrix    13
Name: count, dtype: int64

`groupby("type").head(3)` agrupa por tipo y muestra las primeras 3 filas de cada grupo: sirve para ver un ejemplo de cada tipo de pregunta. Los dobles corchetes `[["qname", ...]]` eligen solo esas columnas.

In [14]:
schema.groupby("type").head(3)[["qname", "type", "question", "sub"]]

,qname,type,question,sub
0,TechEndorse_1,RO,What attracts you to a technology or causes yo...,AI integration or AI Agent capabilities
1,TechEndorse_2,RO,What attracts you to a technology or causes yo...,Easy-to-use API
2,TechEndorse_3,RO,What attracts you to a technology or causes yo...,Robust and complete API
49,MainBranch,MC,Are you someone who writes code? Please select...,NaN
50,Age,MC,What is your age?,NaN
51,EdLevel,MC,Which of the following best describes the high...,NaN
54,WorkExp,TE,How many years of professional work experience...,NaN
59,YearsCode,TE,"Including any education, how many years have y...",NaN
70,ToolCountWork,TE,"For your <b>primary work role</b> (student, de...",NaN
76,Language,Matrix,"Which <b>programming, scripting, and markup la...",NaN


### Variables concretas

`value_counts().head(10)` muestra los 10 valores más frecuentes. Sirve para ver qué categorías tiene `Age` (que es ordinal) y cómo se distribuyen los años programando (`YearsCode`).

In [17]:
df["Age"].value_counts().head(10)



Age
25-34 years old      16519
35-44 years old      13241
18-24 years old       9210
45-54 years old       6275
55-64 years old       2626
65 years or older      942
Prefer not to say      378
Name: count, dtype: int64

In [23]:
df["YearsCode"].value_counts().head(10)

YearsCode
10.0    3108
15.0    2540
20.0    2497
5.0     2267
8.0     2113
6.0     1925
25.0    1832
7.0     1820
12.0    1800
30.0    1740
Name: count, dtype: int64

### Columnas que no coinciden entre `df` y `schema`

Las respuestas y el esquema no tienen exactamente los mismos nombres de columna. Para compararlos se usan **conjuntos** (`set`): `set(A) - set(B)` devuelve los elementos que están en A y no en B.

- `faltan_en_df`: nombres del esquema que no existen como columna en `df`. Son las preguntas tipo matriz (por ejemplo `Language`, `Database`), que en `df` se separan en varias columnas.
- `faltan_en_schema`: columnas de `df` que el esquema no nombra (por ejemplo `LanguageHaveWorkedWith` o `ConvertedCompYearly`).

In [25]:
faltan_en_df = set(schema["qname"]) - set(df.columns)
print(len(faltan_en_df))
print(sorted(faltan_en_df))


13
['AIAgentChallenges', 'AIAgentImpact', 'AIModels', 'AITool', 'CommPlatform', 'Database', 'DevEnvs', 'Language', 'OfficeStackAsync', 'OpSys', 'Platform', 'SOTags', 'Webframe']


In [26]:
faltan_en_schema = set(df.columns) - set(schema["qname"])
print(len(faltan_en_schema))
print(sorted(faltan_en_schema))

46
['AIAgentChallengesNeutral', 'AIAgentChallengesSomewhat agree', 'AIAgentChallengesSomewhat disagree', 'AIAgentChallengesStrongly agree', 'AIAgentChallengesStrongly disagree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat agree', 'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree', 'AIAgentImpactStrongly disagree', 'AIModelsAdmired', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIToolCurrently mostly AI', 'AIToolCurrently partially AI', "AIToolDon't plan to use AI for this task", 'AIToolPlan to mostly use AI', 'AIToolPlan to partially use AI', 'CommPlatformAdmired', 'CommPlatformHaveWorkedWith', 'CommPlatformWantToWorkWith', 'ConvertedCompYearly', 'DatabaseAdmired', 'DatabaseHaveWorkedWith', 'DatabaseWantToWorkWith', 'DevEnvsAdmired', 'DevEnvsHaveWorkedWith', 'DevEnvsWantToWorkWith', 'LanguageAdmired', 'LanguageHaveWorkedWith', 'LanguageWantToWorkWith', 'OfficeStackAsyncAdmired', 'OfficeStackAsyncHaveWorkedWith', 'OfficeStackAsyncWantToWorkWith', 'OpSysPersonal u

### Agrupar columnas por pregunta


In [27]:
prefijos = df.columns.str.replace(r"_\d+.*$", "", regex=True)


In [28]:
conteo = prefijos.value_counts()
conteo.head(30)

JobSatPoints         16
TechEndorse          11
TechOppose           11
SO_Actions           11
ResponseId            1
MainBranch            1
Age                   1
EdLevel               1
Employment            1
EmploymentAddl        1
WorkExp               1
LearnCodeChoose       1
LearnCode             1
LearnCodeAI           1
AILearnHow            1
YearsCode             1
DevType               1
OrgSize               1
ICorPM                1
RemoteWork            1
PurchaseInfluence     1
TechEndorseIntro      1
Industry              1
AIThreat              1
NewRole               1
ToolCountWork         1
ToolCountPersonal     1
Country               1
Currency              1
CompTotal             1
Name: count, dtype: int64

In [29]:
print("Preguntas distintas (prefijos):", prefijos.nunique())
print("De un solo ítem:", (conteo == 1).sum())
print("De varios ítems:", (conteo > 1).sum())

Preguntas distintas (prefijos): 127
De un solo ítem: 123
De varios ítems: 4


### Valores nulos

- `df.isna()` marca con True cada celda vacía. `.sum()` las cuenta por columna y `.mean() * 100` da el porcentaje.
- `sort_values("pct", ascending=False)` ordena de más a menos nulos. `head(20)` muestra las columnas con más nulos y `tail(20)` las que tienen menos.

In [31]:
nulos = df.isna().sum()
pct_nulos = (df.isna().mean() * 100).round(2)

In [34]:
tabla_nulos = pd.DataFrame({"nulos": nulos, "pct": pct_nulos}).sort_values("pct", ascending=False)

tabla_nulos.tail(20)

,nulos,pct
TechEndorse_3,13216,26.87
TechEndorse_2,13216,26.87
TechEndorse_9,13216,26.87
TechEndorse_7,13216,26.87
TechEndorse_1,13216,26.87
TechEndorse_8,13216,26.87
AIThreat,13113,26.66
PurchaseInfluence,11754,23.89
TechEndorseIntro,11635,23.65
WorkExp,6298,12.80


In [33]:
tabla_nulos.head(20)

,nulos,pct
AIAgentObsWrite,48927,99.46
SOTagsWant Entry,48761,99.13
SOTagsHaveEntry,48733,99.07
AIModelsWantEntry,48716,99.03
AIAgentOrchWrite,48713,99.03
JobSatPoints_15_TEXT,48527,98.65
AIAgentKnowWrite,48425,98.44
AIModelsHaveEntry,48415,98.42
SO_Actions_15_TEXT,48368,98.33
AIAgentExtWrite,48332,98.25


### Distribuciones de las variables de las hipótesis

- `describe()` resume una variable numérica (cantidad, media, desvío, mínimo, cuartiles y máximo).
- `skew()` mide la **asimetría**: 0 es simétrica, positivo es una cola larga hacia valores altos. `ConvertedCompYearly` tiene un valor de 75,5: está muy sesgada por unos pocos salarios enormes.
- `value_counts(normalize=True)` devuelve proporciones en lugar de cantidades (0,20 es 20%).

In [46]:
df[["WorkExp", "YearsCode", "ConvertedCompYearly"]].skew()

WorkExp                 1.459810
YearsCode               1.240346
ConvertedCompYearly    75.501796
dtype: float64

In [49]:
df[["WorkExp", "YearsCode", "ConvertedCompYearly"]].describe()


,WorkExp,YearsCode,ConvertedCompYearly
count,42893.000000,43042.000000,2.394700e+04
mean,13.367403,16.570861,1.017615e+05
std,10.800117,11.787610,4.617569e+05
min,1.000000,1.000000,1.000000e+00
25%,5.000000,8.000000,3.817100e+04
50%,10.000000,14.000000,7.532000e+04
75%,20.000000,24.000000,1.205960e+05
max,100.000000,100.000000,5.000000e+07


In [85]:
df["Country"].value_counts(normalize=True).head(15)

Country
United States of America                                0.204109
Germany                                                 0.085363
India                                                   0.071874
United Kingdom of Great Britain and Northern Ireland    0.057623
France                                                  0.039761
Canada                                                  0.036826
Ukraine                                                 0.027203
Poland                                                  0.025059
Netherlands                                             0.024466
Italy                                                   0.023563
Brazil                                                  0.023281
Australia                                               0.022688
Spain                                                   0.020233
Sweden                                                  0.017383
Switzerland                                             0.015408
Name: proportion,

In [84]:
df["AIAcc"].value_counts()

AIAcc
Somewhat trust                9869
Somewhat distrust             8685
Neither trust nor distrust    7162
Highly distrust               6533
Highly trust                  1048
Name: count, dtype: int64

In [83]:
df["NewRole"].value_counts()

NewRole
I have neither consider or transitioned into a new career or industry          16205
I have somewhat considered changing my career and/or the industry I work in    10233
I have strongly considered changing my career and/or the industry I work in     5253
I have transitioned into a new career and/or industry voluntarily               3111
I have transitioned into a new career and/or industry involuntarily              725
Name: count, dtype: int64

# EJERCICIO 2: limpieza

`df_clean = df.copy()` crea una copia independiente de `df`. Todo lo que se limpie se hace sobre `df_clean`, y `df` queda con los datos originales para poder comparar.

In [54]:
df_clean = df.copy()


### Duplicados, identificador y columnas casi vacías

- `df.duplicated().sum()` cuenta las filas completamente repetidas. `df["ResponseId"].is_unique` verifica que no haya dos respuestas con el mismo identificador.
- `pct_nulos[pct_nulos > 90]` deja solo las columnas con más de 90% de nulos, y `.index.tolist()` convierte sus nombres en una lista (`cols_vacias`).

In [59]:
print("Duplicados:", df.duplicated().sum())
print("ResponseId único:", df["ResponseId"].is_unique)

pct_nulos = df_clean.isna().mean() * 100
cols_vacias = pct_nulos[pct_nulos > 90].index.tolist()
print("Columnas con más de 90% de nulos:", len(cols_vacias))
print(cols_vacias)

Duplicados: 0
ResponseId único: True
Columnas con más de 90% de nulos: 30
['TechEndorse_13_TEXT', 'TechOppose_15_TEXT', 'JobSatPoints_15_TEXT', 'LanguagesHaveEntry', 'LanguagesWantEntry', 'DatabaseHaveEntry', 'DatabaseWantEntry', 'PlatformHaveEntry', 'PlatformWantEntry', 'WebframeHaveEntry', 'WebframeWantEntry', 'DevEnvHaveEntry', 'DevEnvWantEntry', 'SOTagsHaveEntry', 'SOTagsWant Entry', 'OfficeStackHaveEntry', 'OfficeStackWantEntry', 'CommPlatformHaveEntr', 'CommPlatformWantEntr', 'AIModelsHaveEntry', 'AIModelsWantEntry', 'SO_Actions_15_TEXT', 'AIAgentImpactStrongly disagree', 'AIAgentKnowledge', 'AIAgentKnowWrite', 'AIAgentOrchestration', 'AIAgentOrchWrite', 'AIAgentObserveSecure', 'AIAgentObsWrite', 'AIAgentExtWrite']


Control: ninguna de las variables de las hipótesis debe estar entre las columnas casi vacías. La lista tiene que salir `[]`.

In [60]:
vars_hip = ["Country","Currency","CompTotal","ConvertedCompYearly","WorkExp","AIAcc","JobSat","NewRole"]
print([v for v in vars_hip if v in cols_vacias])   # debería dar []

[]


### Formato de `Currency`

`str.match(r"^[A-Z]{3}")` pregunta si el texto empieza con 3 letras mayúsculas (el código de la moneda, por ejemplo `ARS`). `str.contains("\t")` busca tabulaciones: cerca de un cuarto de los valores tiene la forma `ARS<tab>Argentine peso`, el resto tiene `EUR European Euro`. También se mira el nombre de las columnas `AIAgentImpact...` y cuántas respuestas tienen las columnas de agentes.

In [64]:
# Acción 3: formato de Currency
cur = df_clean["Currency"].dropna()
print("Todos empiezan con 3 letras mayúsculas:", cur.str.match(r"^[A-Z]{3}").all())
print("Proporción con tabulación:", cur.str.contains("\t").mean())
print(cur[~cur.str.contains("\t")].value_counts().head())

# Nombre raro
print([c for c in df.columns if c.startswith("AIAgentImpact")])

# Cuántas respuestas tienen las 3 columnas MC de agentes
print(df["AIAgentKnowledge"].notna().sum(),
      df["AIAgentOrchestration"].notna().sum(),
      df["AIAgentObserveSecure"].notna().sum())

Todos empiezan con 3 letras mayúsculas: False
Proporción con tabulación: 0.24736856957417389
Currency
EUR European Euro           9618
USD United States dollar    8780
INR Indian rupee            2484
GBP Pound sterling          2047
CAD Canadian dollar         1283
Name: count, dtype: int64
['AIAgentImpactSomewhat agree', 'AIAgentImpactNeutral', 'AIAgentImpactSomewhat disagree', 'AIAgentImpactStrongly agree', 'AIAgentImpactStrongly disagree']
3401 3760 2692


### Apartar las columnas casi vacías

`df_clean.drop(columns=cols_vacias)` elimina esas columnas de `df_clean`, pero antes se guardan en `df_apartadas` para no perderlas. Se apartan en lugar de borrarlas del todo.

In [65]:
df_apartadas = df_clean[cols_vacias].copy()
df_clean = df_clean.drop(columns=cols_vacias)
print(df_clean.shape, df_apartadas.shape)

(49191, 142) (49191, 30)


`pd.concat([..., ...], axis=1)` pega tablas lado a lado (por columnas). Acá agrega a `df_apartadas` cuatro columnas más de `AIAgentImpact...`, y después se sacan de `df_clean`.

In [70]:
imp_restantes = ["AIAgentImpactSomewhat agree", "AIAgentImpactNeutral",
                 "AIAgentImpactSomewhat disagree", "AIAgentImpactStrongly agree"]

df_apartadas = pd.concat([df_apartadas, df_clean[imp_restantes]], axis=1)
df_clean = df_clean.drop(columns=imp_restantes)
print(df_clean.shape, df_apartadas.shape)

(49191, 138) (49191, 34)


### Separar `Currency` en código y descripción

- `str.split(r"\s+", n=1)` corta el texto en el primer espacio o tabulación (`\s+`) en dos partes, y `.str[1]` se queda con la segunda (el nombre de la moneda).
- `str.extract(r"^([A-Z]{3})")` extrae solo las 3 letras mayúsculas del principio (el código). Lo que va entre paréntesis es lo que se extrae.
- Se leen desde `df` (el original) y no desde `df_clean`: así la celda se puede volver a ejecutar sin que falle ni deje la columna vacía.
- `extract` devuelve nulo cuando el texto no empieza con 3 mayúsculas. En el resultado se ve que `Currency` pasa de 13.754 a 13.757 nulos: hay 3 valores que no tienen ese formato y quedaron nulos.

In [73]:
df_clean["Currency_desc"] = df["Currency"].str.split(r"\s+", n=1, regex=True).str[1]
df_clean["Currency"] = df["Currency"].str.extract(r"^([A-Z]{3})", expand=False)

print(df_clean["Currency"].isna().sum(), df["Currency"].isna().sum())
print(df_clean[["Currency", "Currency_desc"]].dropna().head())
print((df_clean["Currency"] == "ARS").sum(), (df_clean["Currency"] == "USD").sum())

13757 13754
  Currency         Currency_desc
0      EUR         European Euro
1      EUR         European Euro
2      UAH     Ukrainian hryvnia
3      EUR         European Euro
4      USD  United States dollar
142 8780


`str.contains("Argentin", na=False)` da True para las filas de Argentina (con `na=False` los nulos cuentan como False). `df_clean.loc[arg, "Currency"]` toma la columna `Currency` solo de esas filas. Se ve cuántos argentinos reportaron en pesos (ARS) y cuántos en dólares (USD).

In [74]:
arg = df_clean["Country"].str.contains("Argentin", na=False)
print(df_clean.loc[arg, "Currency"].value_counts())

Currency
ARS    142
USD     80
Name: count, dtype: int64


In [75]:
print(df_clean.shape, df_apartadas.shape)

(49191, 139) (49191, 34)


### Edad ordenada

- `pd.Categorical(..., categories=orden_edad, ordered=True)` crea una variable categórica con un orden explícito (de menor a mayor edad). Sin esto pandas ordenaría alfabéticamente.
- `df["Age"].where(df["Age"] != "Prefer not to say")` deja el valor de edad donde la condición es verdadera y pone nulo en el resto: "Prefer not to say" no es un grupo de edad, así que pasa a nulo (378 personas).
- `value_counts().sort_index()` cuenta cada grupo y los muestra en el orden de la categoría.

In [82]:
orden_edad = ["18-24 years old", "25-34 years old", "35-44 years old",
              "45-54 years old", "55-64 years old", "65 years or older"]

df_clean["Age_ord"] = pd.Categorical(
    df["Age"].where(df["Age"] != "Prefer not to say"),
    categories=orden_edad,
    ordered=True
)

print(df_clean["Age_ord"].isna().sum())
print(df_clean["Age_ord"].value_counts().sort_index())

378
Age_ord
18-24 years old       9210
25-34 years old      16519
35-44 years old      13241
45-54 years old       6275
55-64 years old       2626
65 years or older      942
Name: count, dtype: int64
